In [ ]:
import numpy as np
import pandas as pd

## Lecture des données

In [ ]:
permis_df = pd.read_csv('statistiques-permis-de-construction.csv')
permis_df

## Nettoyage des données
### Validation des arrondissements

In [ ]:
arrondissements = permis_df['arrondissement'].unique()
arrondissements

* Obtenir les identifiants des arrondissements

In [ ]:
arrond_id = pd.read_csv(
    'mtl_arrondissements.csv',
    index_col='Arrondissement'
)['arrond_id']

arrond_id

* Tester les arrondissements de `permis_df`

In [ ]:
for nom_arrond in arrondissements:
    if nom_arrond in arrond_id:
        print(nom_arrond, arrond_id[nom_arrond])
    else:
        print('* ERREUR:', arrond_id, 'absent')

* Convertir l'arrondissement en identifiants

In [ ]:
if 'arrondissement' in permis_df.columns:
    permis_df['arrondissement'] = permis_df['arrondissement'].apply(
        lambda nom_arrond: arrond_id[nom_arrond])
    permis_df.rename(
        columns={'arrondissement': 'arrond_id'}, inplace=True)

permis_df

### Remplissage des valeurs non définies
* Trouver des coûts de permis non définis

In [ ]:
permis_df[permis_df['cout_permis_emis'].isna()]

* Trouver les enregistrements de 1990 ayant un code `CO` ou `DE`

In [ ]:
permis_1990_CO_DE = permis_df[
    (permis_df['annee'] == 1990) &
    permis_df['code_type_base_demande'].isin(['CO', 'DE'])
]

permis_1990_CO_DE

* Calculer un coût moyen de permis pour l'enregistrement ci-haut

In [ ]:
sommes = permis_1990_CO_DE.sum()
permis_df.loc[29, 'cout_permis_emis'] = np.round(
    sommes['cout_permis_emis'] / sommes['nombre_permis_emis'],
    decimals=2
)

permis_df.loc[29, :]

* Trouver des coûts des travaux non définis

In [ ]:
permis_df[permis_df['cout_travaux_estimes'].isna()]

In [ ]:
permis_df[
    (permis_df['code_type_base_demande'] == 'DE') &
    ~permis_df['cout_travaux_estimes'].isna()
]

* Pas assez de données pour reconstruire les valeurs manquantes

## Analyse sommaire
* Coût total des permis par année et par arrondissement

In [ ]:
cout_total_permis = permis_df.pivot_table(
    values='cout_permis_emis', aggfunc='sum',
    index='annee', columns='arrond_id'
) / 1000000  # M$

cout_total_permis

In [ ]:
cout_total_permis.plot(kind='line')

In [ ]:
cout_total_travaux = permis_df.pivot_table(
    values='cout_travaux_estimes', aggfunc='sum',
    index='annee', columns='arrond_id'
) / 1000000  # M$

cout_total_travaux

In [ ]:
cout_total_travaux.plot(kind='line')

In [ ]:
permis_df.to_csv('construction_permis.csv', index=False)